# Bronze Cleanup — Per-column type inference for `brfss_2024`

The bronze table `data_engineering.bronze.brfss_2024` was ingested with all 301 columns typed as `string`,
even though most columns hold numeric content. That breaks downstream math (`x / 100`, `np.log1p(...)`,
label maps keyed on `1.0`/`2.0`).

This notebook does a **best-effort cast per column**:

| Signal in the data | Decision |
|---|---|
| Contains `.` anywhere | `double` |
| Starts with `0` then another digit (leading-zero padding, e.g. `"02"`, `"02282024"`) | `string` (preserve) |
| All values cast cleanly to bigint | `bigint` |
| All values cast cleanly to double, not int | `double` |
| Any value can't be parsed as a number | `string` (fallback) |

The plan is built from a 100k-row sample for speed, then validated against the **full** table after casting —
any column whose null count grows is reverted to `string` before the final write.

In [0]:
import re
import pandas as pd
import numpy as np
from collections import Counter
from pyspark.sql import functions as F

SRC = "data_engineering.bronze.brfss_2024"
SAMPLE_SIZE = 100_000

df_src = spark.read.table(SRC)
n_rows = df_src.count()
print(f"Source: {SRC}")
print(f"Shape:  {n_rows:,} rows x {len(df_src.columns)} cols")

Source: data_engineering.bronze.brfss_2024
Shape:  457,670 rows x 301 cols


## 1. Infer per-column type from a 100k-row sample

In [0]:
sample_pdf = df_src.limit(SAMPLE_SIZE).toPandas()

def infer_type(series: pd.Series) -> str:
    s = series.dropna().astype(str)
    if len(s) == 0:
        return "string"   # all null — nothing to base a decision on
    has_dot         = s.str.contains(r"\.",       regex=True).any()
    has_leading_zero = s.str.contains(r"^0[0-9]", regex=True).any()
    numeric          = pd.to_numeric(s, errors="coerce")
    if numeric.isna().any():
        return "string"   # at least one value is non-numeric → fall back
    if has_dot:
        return "double"
    if has_leading_zero:
        return "string"   # zero-padded codes/dates — keep formatting
    # All non-null values are integer-shaped
    return "bigint"

plan = {c: infer_type(sample_pdf[c]) for c in sample_pdf.columns}
counts = Counter(plan.values())
print("Inferred plan:")
for t, n in counts.most_common():
    print(f"  {t:<8} {n:>3} columns")

In [0]:
# Examples per inferred type
for t in ("bigint", "double", "string"):
    cols_of_t = [c for c, x in plan.items() if x == t]
    print(f"\n{t.upper()} — {len(cols_of_t)} columns")
    for c in cols_of_t[:15]:
        ex = sample_pdf[c].dropna().head(3).tolist()
        print(f"  {c:<15} examples: {ex}")
    if len(cols_of_t) > 15:
        print(f"  ... and {len(cols_of_t) - 15} more")

## 2. Apply the cast plan with `try_cast`

In [0]:
def build_select(plan: dict):
    exprs = []
    for c, t in plan.items():
        if t == "string":
            exprs.append(F.col(f"`{c}`"))
        else:
            exprs.append(F.expr(f"try_cast(`{c}` as {t}) as `{c}`"))
    return exprs

df_cast = df_src.select(*build_select(plan))

## 3. Validate on the **full** table — revert any column whose nulls grew

In [0]:
null_before = df_src.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_src.columns]
).first().asDict()
null_after = df_cast.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_cast.columns]
).first().asDict()

regressions = [c for c in plan if null_after[c] > null_before[c]]

if regressions:
    print(f"WARNING — {len(regressions)} columns gained nulls after cast; reverting to string:")
    for c in regressions:
        delta = null_after[c] - null_before[c]
        print(f"  {c:<15} {plan[c]:<7} -> string   (+{delta:,} nulls would have been introduced)")
        plan[c] = "string"
    # Rebuild df_cast with the corrected plan
    df_cast = df_src.select(*build_select(plan))
else:
    print(f"OK — all {len(plan)} columns cast cleanly; no new nulls introduced.")

final_counts = Counter(plan.values())
print("\nFinal plan:")
for t, n in final_counts.most_common():
    print(f"  {t:<8} {n:>3} columns")

## 4. Preview a few representative columns (before vs after)

In [0]:
preview_cols = [c for c in [
    "_STATE", "IMONTH", "IDATE", "SEQNO", "_PSU", "_BMI5", "_LLCPWT", "GENHLTH", "CVDINFR4"
] if c in df_src.columns]

print("Inferred types for these columns:")
for c in preview_cols:
    print(f"  {c:<10} {plan[c]}")

print("\nBEFORE (all string):")
df_src.select(*preview_cols).show(3, truncate=False)
print("AFTER:")
df_cast.select(*preview_cols).show(3, truncate=False)

## 5. Overwrite bronze table

In [0]:
(df_cast.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SRC))

after = spark.read.table(SRC)
print(f"Wrote {SRC}")
print(f"  Rows:  {after.count():,}")
print(f"  Cols:  {len(after.columns)}")
print(f"  Types: {dict(Counter(t for _, t in after.dtypes))}")